In [ ]:
# Adopted from LDM's KL-VAE: https://github.com/CompVis/latent-diffusion
import numpy as np
import torch
import torch.nn as nn


def topk(x: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Creates a mask where values are 0 for the top-k elements in each batch and 1 elsewhere.

    Args:
        x : tensor of shape [B, L] containing values to find top-k elements
        k : tensor of shape [B, 1] containing the number of top elements to find for each batch

    Returns:
        final_mask: tensor of shape [B,L] where final_mask[b,i] = 0 if x[b,i] is in
                   the k[b] biggest values of x[b,:], else final_mask[b,i] = 1
    """
    B, L = x.shape  # batchsize, list size

    # Get indices sorted in descending order
    _, indices_des = torch.sort(x, dim=-1, descending=True)

    # Create range mask [1, L] and repeat it B times
    mask = torch.arange(L, device=x.device).unsqueeze(0).expand(B, -1)
    k_expanded = k.expand(-1, L)
    mask = mask < k_expanded

    # Create one-hot encoding and apply mask
    one_hot = torch.nn.functional.one_hot(indices_des, num_classes=L).float()
    one_hot = one_hot * mask.unsqueeze(-1)

    # Sum along the appropriate dimension to get final mask
    final_mask = one_hot.sum(dim=1)

    # Flip the mask (0 for top-k, 1 for others)
    return final_mask


def convert_to_counts_batched(probs: torch.Tensor, N: torch.Tensor) -> torch.Tensor:
    """Convert logits to counts by flooring the probabilities and then distributing
    the remaining counts to the top decimal parts.

    Args:
        logits_batch (torch.Tensor): batch of logits of shape [B, L]
        N (torch.Tensor): number of counts to use to convert logits to a histogram,
        accepts shape [B] or [B, 1]

    Returns:
        torch.Tensor: counts of shape [B, L]

    Examples:
        >>> import torch
        >>> N = torch.tensor([5,5,5,4])
        >>> test_case = torch.tensor([
        ...     [1., 1., 1., 1., 1., 1., 1., 1.],  # All equal
        ...     [2., 1., 1., 1., 1., 1., 1., 1.],  # First value different
        ...     [2., 2., 1., 0., 0., 0., 0., 0.],  # Two twos, one one
        ...     [100., 100., 100., -100., -100., -100., -100., -100.]  # Extreme values
        ... ])
        >>> result = convert_to_counts_batched(test_case, N)
        >>> expected = torch.tensor([
        ...     [1, 1, 1, 1, 1, 0, 0, 0],
        ...     [1, 1, 1, 1, 1, 0, 0, 0],
        ...     [2, 2, 1, 0, 0, 0, 0, 0],
        ...     [2, 1, 1, 0, 0, 0, 0, 0]
        ... ])
        >>> (result == expected).all().item()
        True
        >>> (result.sum(dim=-1) == N).all().item()
        True
    """
    N = N.reshape(-1, 1)
    # Expect logits_batch shape: (batch_size, vocab_size)
    counts = torch.floor(probs * N).long()
    remaining = N - counts.sum(dim=-1, keepdim=True)  # Shape: (batch_size, 1)

    decimal_parts = (probs * N) - counts.float()
    # Get indices of top decimal parts for each batch
    top_k_decimal_mask = topk(decimal_parts, remaining).to(torch.bool)

    # Increment top k in parallel
    counts[top_k_decimal_mask] += 1

    return counts


class HistogramNormalizer(torch.nn.Module):
    def __init__(self, h_token, o_token, vocab_size, num_bits=16, scale=1):
        super().__init__()
        self.h_token = h_token
        self.o_token = o_token
        self.vocab_size = vocab_size
        self.num_bits = num_bits
        self.scale = scale
        # Create powers of 2 as a buffer to avoid recomputing
        self.register_buffer("powers", torch.pow(2, torch.arange(num_bits - 1, -1, -1).float()))

    def get_normalized_size(self):
        return self.vocab_size * self.num_bits

    def count_to_binary(self, count):
        """Convert number(s) to binary representation using PyTorch operations."""
        return ((count.unsqueeze(-1) // self.powers) % 2).to(torch.int)

    def binary_to_count(self, binary):
        return (binary * self.powers).sum(dim=-1, keepdim=True)

    def transform(self, x):
        # Zero out special tokens
        x[:, self.h_token] = 0
        x[:, self.o_token] = 0

        # Get counts and normalize histogram
        b, _ = x.shape

        # Convert counts to binary representation
        data = self.count_to_binary(x.reshape(-1)).reshape(b, -1).to(torch.float32)

        # Shift from [0,1] to [-self.scale,self.scale] range
        return (data * 2 - 1) * self.scale

    def reverse_transform(self, x):
        # Shift from [-self.scale,self.scale] to [0,1]  range
        x = (x / self.scale + 1) / 2

        # Clip to [0,1] range
        x = x.clip(min=0, max=1).round().int()
        # Split into histogram and binary count
        b, _ = x.shape

        # Convert from batch_size x (vocab_size * num_bits) to (batch_size x vocab_size) x num_bits
        x = x.reshape(b * self.vocab_size, -1)
        # Convert binary back to count, and reshape
        x = self.binary_to_count(x)
        # Convert from (batch_size x vocab_size) x 1 to batch_size x vocab_size
        x = x.reshape(b, -1)

        # Restore special tokens
        x[:, self.h_token] = 1
        x[:, self.o_token] = 1

        return x, x.sum(dim=-1)


from torchvision.ops import MLP


class VAEEncoder(nn.Module):
    def __init__(
        self,
        input_dim=3,
        latent_dim=16,
        hidden_dims=[64, 32],
        dropout=0.0,
    ):
        super().__init__()
        # Create MLP encoder using torchvision
        self.encoder = MLP(
            in_channels=input_dim,
            hidden_channels=hidden_dims,
            norm_layer=nn.LayerNorm,
            activation_layer=nn.SiLU,
            dropout=dropout,
            inplace=None,  # Explicit None for clarity
        )
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_var = nn.Linear(hidden_dims[-1], latent_dim)

    def forward(self, x):
        result = self.encoder(x)
        mu = self.fc_mu(result)
        log_var = self.fc_var(result)
        return mu, log_var


class VAEDecoder(nn.Module):
    def __init__(
        self,
        output_dim=3,
        latent_dim=16,
        hidden_dims=[32, 64],
        dropout=0.0,
    ):
        super().__init__()
        self.decoder = MLP(
            in_channels=latent_dim,
            hidden_channels=hidden_dims + [output_dim],
            norm_layer=nn.LayerNorm,
            activation_layer=nn.SiLU,
            dropout=dropout,
            inplace=None,  # Explicit None for clarity
        )

    def forward(self, z):
        return self.decoder(z)


class DiagonalGaussianDistribution(object):
    def __init__(self, parameters, deterministic=False):
        self.parameters = parameters
        self.mean, self.logvar = torch.chunk(parameters, 2, dim=1)
        self.logvar = torch.clamp(self.logvar, -30.0, 20.0)
        self.deterministic = deterministic
        self.std = torch.exp(0.5 * self.logvar)
        self.var = torch.exp(self.logvar)
        if self.deterministic:
            self.var = self.std = torch.zeros_like(self.mean).to(device=self.parameters.device)

    def sample(self):
        x = self.mean + self.std * torch.randn(self.mean.shape).to(device=self.parameters.device)
        return x

    def kl(self, other=None):
        if self.deterministic:
            return torch.Tensor([0.0])
        else:
            if other is None:
                return 0.5 * torch.sum(
                    torch.pow(self.mean, 2) + self.var - 1.0 - self.logvar,
                    dim=[1],
                )
            else:
                return 0.5 * torch.sum(
                    torch.pow(self.mean - other.mean, 2) / other.var
                    + self.var / other.var
                    - 1.0
                    - self.logvar
                    + other.logvar,
                    dim=[1],
                )

    def nll(self, sample, dims=[1]):
        if self.deterministic:
            return torch.Tensor([0.0])
        logtwopi = np.log(2.0 * np.pi)
        return 0.5 * torch.sum(
            logtwopi + self.logvar + torch.pow(sample - self.mean, 2) / self.var,
            dim=dims,
        )

    def mode(self):
        return self.mean


class AutoencoderKL(nn.Module):
    def __init__(self, embed_dim, input_dim, use_variational=True, ckpt_path=None):
        super().__init__()
        assert use_variational
        self.use_variational = use_variational
        self._encoder = VAEEncoder(input_dim=input_dim, latent_dim=embed_dim)
        self._decoder = VAEDecoder(output_dim=input_dim, latent_dim=embed_dim)
        self.embed_dim = embed_dim
        self.beta = 1

    def encode(self, x):
        mu, log_var = self._encoder(x)
        moments = torch.cat((mu, log_var), 1)
        posterior = DiagonalGaussianDistribution(moments)
        return posterior

    def decode(self, z):
        dec = self._decoder(z)
        return dec

    def forward(self, inputs, disable=False):
        return self.training_step(inputs, disable)

    def training_step(self, inputs, disable=False):
        """
        Training step for the VAE model.

        Args:
            inputs: Input tensor
            disable: Flag to disable parts of the loss computation
            optimizer_idx: Index of the optimizer (not used in this implementation)

        Returns:
            dict: Dictionary containing loss components and reconstructed output
        """
        posterior = self.encode(inputs)

        if disable:
            # Deterministic warmup: directly use mean
            z = posterior.mean
        else:
            # Sample from posterior
            z = posterior.sample()

        # Decode the latent representation
        dec = self.decode(z)

        # Compute reconstruction loss (mean squared error)
        rec_loss = torch.nn.functional.mse_loss(dec, inputs, reduction="mean")

        # Compute KL divergence loss if using variational mode
        kl_loss = torch.zeros_like(rec_loss)
        if self.use_variational and not disable:
            kl_loss = posterior.kl().mean()

        # Total loss is reconstruction loss + KL divergence
        loss = rec_loss + self.beta * kl_loss

        return {
            "loss": loss,
            "rec_loss": rec_loss,
            "kl_loss": kl_loss,
            "reconstruction": dec,
        }


vocab_size = 6
n_bits = 8
histogram_normalizer = HistogramNormalizer(4, 5, vocab_size, n_bits, scale=1)

histogram = torch.tensor([[0, 4, 3, 1, 1, 1]], dtype=torch.float32)
data = histogram_normalizer.transform(histogram)
vae = AutoencoderKL(embed_dim=4, input_dim=histogram_normalizer.get_normalized_size())

# Training parameters
optimizer = torch.optim.AdamW(vae.parameters(), lr=1e-2, weight_decay=0.01)
num_epochs = 50
losses = []

# Training loop
for epoch in range(num_epochs):
    optimizer.zero_grad()

    # Forward pass
    outputs = vae(data)
    loss = outputs["loss"]
    losses.append(loss.item())
    # Backward pass and optimize
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

import matplotlib.pyplot as plt

plt.plot(losses)

# Test the trained model
with torch.no_grad():
    encoded = vae.encode(data).mean
    decoded = vae.decode(encoded)
    output, _ = histogram_normalizer.reverse_transform(decoded)

    # Print results
    print("\nOriginal histogram:", histogram.reshape(-1).numpy())
    print("Reconstructed histogram:", output.reshape(-1).numpy().round())

In [ ]:
from types import SimpleNamespace

import matplotlib.pyplot as plt
import torch
import torch.optim as optim
from omegaconf import DictConfig
from torch.nn.utils import clip_grad_norm_
from x_transformers import Decoder, TransformerWrapper

from meds_torch.input_encoder import INPUT_ENCODER_MASK_KEY, INPUT_ENCODER_TOKENS_KEY
from meds_torch.models.diffusion_utils.diffloss import DiffLoss


def get_diffusion_loss(self, prompts, histogram, embeddings, mask, histogram_normalizer):
    # All inputs except the last we can evaluate
    prompts = prompts[:, :-1]  # ignore last h token
    embeddings = embeddings[:, :-1]  # ignore last h token

    # Ground truth histogram is shifted by one, as we are predicting the next histogram
    histogram = histogram[:, 1:]
    mask = mask[:, 1:].to(torch.bool)  # last unmasked token has invalid histogram so mask it

    h_mask = (prompts == self.cfg.h_token) & mask
    patch_embeddings = embeddings[h_mask]
    num_diffusion_samples = h_mask.sum()

    # Setup target -- histogram + counts
    target = histogram[h_mask, :]
    target = histogram_normalizer.transform(target)
    return target, patch_embeddings.reshape(num_diffusion_samples, -1)


# Training loop
def train(autoencoder, diffusion, histogram_normalizer, num_epochs=20, log_interval=1, lr=1e-3):
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    autoencoder = autoencoder.to(device)
    diffusion = diffusion.to(device)
    histogram_normalizer.to(device)

    autoencoder.train()
    diffusion.train()
    losses = []
    avg_losses = []
    mse_losses = []
    loss_dicts = []

    # Initialize optimizer
    optimizer = optim.AdamW(
        list(diffusion.parameters()) + list(autoencoder.parameters()), lr=lr, weight_decay=0.01
    )
    scheduler = optimizer
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

    def train_step(code, histogram, z, mask, self):
        # Move data to device
        code = code.to(device)
        histogram = histogram.to(device)
        z = z.to(device)
        mask = mask.to(device)

        # Zero gradients
        scheduler.zero_grad()

        # Get diffusion inputs
        target, patch_embeddings = get_diffusion_loss(self, code, histogram, z, mask, histogram_normalizer)
        with torch.no_grad():
            embedding_target = autoencoder.encode(target).sample()
        # Forward pass and loss computation
        loss = diffusion.forward(
            embedding_target.detach().repeat(self.cfg.diffusion_batch_mul, 1),
            patch_embeddings.repeat(self.cfg.diffusion_batch_mul, 1),
        )
        loss_dict = autoencoder.forward(target)
        loss_dict["vae_loss"], loss_dict["diffusion_loss"], loss_dict["loss"] = (
            loss_dict["loss"],
            loss,
            loss + loss_dict["loss"],
        )

        loss = loss_dict["loss"]

        # Backward pass
        loss.backward()

        # Gradient clipping
        # clip_grad_norm_(diffusion.parameters(), max_norm=1.0)find

        # Optimizer step
        optimizer.step()

        return {k: v.item() for k, v in loss_dict.items() if k != "reconstruction"}, target, patch_embeddings

    for epoch in range(num_epochs):
        # Your data loading/preparation code here
        code = torch.tensor(
            [[4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4]]
        )
        mask = torch.ones_like(code, dtype=torch.bool)
        z = (
            torch.randn(1, code.shape[1], embedding_size) * 0.1
        )  # torch.zeros(1, code.shape[1], embedding_size)

        histogram = torch.tensor(
            [
                [
                    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
                    [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
                    [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
                    [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
                    [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
                    [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
                    [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
                    [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
                ]
            ]
        )

        # Create namespace for configuration
        self = SimpleNamespace()
        self.cfg = SimpleNamespace()
        self.cfg.h_token = 4
        self.cfg.o_token = 5
        self.cfg.diffusion_batch_mul = 160
        # Training step
        loss_dict, target, patch_embeddings = train_step(code, histogram, z, mask, self)
        loss_dicts.append(loss_dict)
        losses.append(loss_dict["loss"])

        # Step the learning rate scheduler
        scheduler.step()

        # Logging
        if (epoch + 1) % log_interval == 0:
            avg_loss = sum(losses[-log_interval:]) / log_interval
            print(f"Epoch [{epoch+1}/{num_epochs}], Average Loss: {avg_loss:.4f}")
            avg_losses.append(avg_loss)
            diffusion.eval()
            with torch.inference_mode():
                sample = diffusion.sample(patch_embeddings, temperature=1.0)
                sample = autoencoder.decode(sample)
                mse_loss = (target - sample).pow(2).mean()
                print(f"MSE Loss: {mse_loss:.4f}")
                mse_losses.append(mse_loss.item())
            diffusion.train()
    return loss_dicts, avg_losses, mse_losses, diffusion, target, patch_embeddings


# Model and data setup
n_bits = 8
vocab_size = 6
histogram_normalizer = HistogramNormalizer(4, 5, vocab_size, n_bits, scale=1)

normalized_histogram_size = histogram_normalizer.get_normalized_size()
embedding_size = 7
target_embedding_size = 32
diffloss_w = 768
diffloss_d = 6
num_sampling_steps = "100"
diffusion = DiffLoss(
    target_channels=target_embedding_size,
    z_channels=embedding_size,
    width=diffloss_w,
    depth=diffloss_d,
    num_sampling_steps=num_sampling_steps,
    grad_checkpointing=False,
    noise_schedule="cosine",
)
autoencoder = AutoencoderKL(embed_dim=target_embedding_size, input_dim=normalized_histogram_size)


# Run training
loss_dicts, losses, mse_losses, diffusion, target, patch_embeddings = train(
    autoencoder, diffusion, histogram_normalizer, num_epochs=50
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"min(losses): {min(losses)}")
if len(mse_losses):
    print(f"min(mse_losses): {min(mse_losses)}")
z = patch_embeddings.to(device)[:1, :]
diffusion.to(device)
diffusion.eval()
with torch.inference_mode():
    sample = diffusion.sample(z, temperature=1.0)
    sample = autoencoder.decode(sample)
x, counts = histogram_normalizer.reverse_transform(sample)
print(f"raw target: {target[0]}")
print(f"raw sample: {sample}")
print(f"counts: {counts}")
print(f"sample: {x}")
print(f"target: {[0., 4., 3., 1., 1., 1.]}")


# test = torch.tensor([[0., 4., 3., 1., 1., 1.]]).to(device)
# print(test)
# print(histogram_normalizer.transform(test))
# print(histogram_normalizer.reverse_transform(histogram_normalizer.transform(test)))

In [ ]:
z = patch_embeddings.to(device)[:1, :]
diffusion.to(device)
diffusion.eval()
with torch.inference_mode():
    sample = diffusion.sample(z, temperature=1.0)
    sample = autoencoder.decode(sample)
x, counts = histogram_normalizer.reverse_transform(sample)
print(f"raw target: {target[0]}")
print(f"raw sample: {sample}")
print(f"counts: {counts}")
print(f"sample: {x}")
print(f"target: {[0., 4., 3., 1., 1., 1.]}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_training_metrics(loss_dicts, mse):
    # Extract metrics and convert to numpy arrays
    metrics = {}
    for key in loss_dicts[0].keys():
        metrics[key] = np.array([d[key] for d in loss_dicts])
    metrics["mse"] = np.array(mse)

    # Calculate number of metrics and setup subplot grid
    n_metrics = len(metrics)
    n_cols = 2
    n_rows = (n_metrics + 1) // 2  # Ceiling division

    # Create figure and subplots
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)

    # Plot each metric
    for idx, (metric_name, values) in enumerate(metrics.items()):
        row = idx // 2
        col = idx % 2

        ax = axes[row, col]
        steps = np.arange(1, len(values) + 1)

        # Plot with styling
        ax.plot(steps, values, "b-", linewidth=2, alpha=0.7)
        ax.set_title(f"{metric_name} over Time", fontsize=12, pad=10)
        ax.set_xlabel("Training Step", fontsize=10)
        ax.set_ylabel(metric_name, fontsize=10)
        ax.grid(True, linestyle="--", alpha=0.7)

        # Add some padding to the y-axis limits
        y_min, y_max = values.min(), values.max()
        y_range = y_max - y_min
        ax.set_ylim(y_min - 0.1 * y_range, y_max + 0.1 * y_range)

    # If odd number of metrics, remove the empty subplot
    if n_metrics % 2 == 1:
        fig.delaxes(axes[-1, -1])

    # Adjust layout to prevent overlap
    plt.tight_layout()
    return fig


# Example usage:
fig = plot_training_metrics(loss_dicts, mse_losses)
plt.show()

In [ ]:
pred, counts = histogram_normalizer.reverse_transform(sample)
print(f"diff: {pred.to("cpu") - torch.tensor([[0., 4., 3., 1., 1., 1.]])}")
# print(f"raw target: {target[0]}")
# print(f"raw sample: {sample}")
print(target[0, :6])
x = sample[0, :6]
print(x)
(x + 1) / 2 * 8

In [ ]:
print(f"raw target: {target[0]}")

In [ ]:
def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    """
    Create a beta schedule that discretizes the given alpha_t_bar function,
    which defines the cumulative product of (1-beta) over time from t = [0,1].
    :param num_diffusion_timesteps: the number of betas to produce.
    :param alpha_bar: a lambda that takes an argument t from 0 to 1 and
                      produces the cumulative product of (1-beta) up to that
                      part of the diffusion process.
    :param max_beta: the maximum beta to use; use values lower than 1 to
                     prevent singularities.
    """
    betas = []
    alpha_bar_values = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        alpha_bar_values.append(alpha_bar(t1))
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas) * 0.02, np.array(alpha_bar_values)


import math

import numpy as np

x, alpha_bar_values_1 = betas_for_alpha_bar(
    1000, lambda t: math.cos((t + 0.0002) / 1.00025 * math.pi / 2) ** 2
)
plt.plot(x, label="beta_1")
x, alpha_bar_values_2 = betas_for_alpha_bar(1000, lambda t: math.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2)
plt.plot(x, label="beta_2")
x, alpha_bar_values_3 = betas_for_alpha_bar(1000, lambda t: math.cos((t + 0.5) / 1.5 * math.pi / 2) ** 2)
plt.plot(x, label="beta_3")

plt.plot(np.linspace(0.0001, 0.02, 1000, dtype=np.float64), label="linear")

plt.legend()
plt.show()


plt.plot(alpha_bar_values_1, label="alpha_bar_1")
plt.plot(alpha_bar_values_2, label="alpha_bar_2")
plt.plot(alpha_bar_values_3, label="alpha_bar_3")
plt.legend()
plt.show()

# noise = np.arange(1000)
# next_noise = noise + 1
# def noise_schedule_func(t):
#     return np.cos((t + 0.008) / 1.008 * np.pi / 2) ** 2

# noise = noise_schedule_func(noise)
# next_noise = noise_schedule_func(next_noise)
# print(noise)


# plt.plot(noise)

In [ ]:
import numpy as np

t = np.linspace(0, 1, 1001)
alpha_bar = np.cos((t + 0.008) / 1.008 * np.pi / 2) ** 2
beta = 1 - alpha_bar[1:] / alpha_bar[:-1]

plt.plot(t[:-1], beta)

In [ ]:
def get_beta_schedule(beta_schedule, *, beta_start, beta_end, num_diffusion_timesteps):
    """
    This is the deprecated API for creating beta schedules.
    See get_named_beta_schedule() for the new library of schedules.
    """
    if beta_schedule == "quad":
        betas = (
            np.linspace(
                beta_start**0.5,
                beta_end**0.5,
                num_diffusion_timesteps,
                dtype=np.float64,
            )
            ** 2
        )
    elif beta_schedule == "linear":
        betas = np.linspace(beta_start, beta_end, num_diffusion_timesteps, dtype=np.float64)
    elif beta_schedule == "warmup10":
        betas = _warmup_beta(beta_start, beta_end, num_diffusion_timesteps, 0.1)
    elif beta_schedule == "warmup50":
        betas = _warmup_beta(beta_start, beta_end, num_diffusion_timesteps, 0.5)
    elif beta_schedule == "const":
        betas = beta_end * np.ones(num_diffusion_timesteps, dtype=np.float64)
    elif beta_schedule == "jsd":  # 1/T, 1/(T-1), 1/(T-2), ..., 1
        betas = 1.0 / np.linspace(num_diffusion_timesteps, 1, num_diffusion_timesteps, dtype=np.float64)
    else:
        raise NotImplementedError(beta_schedule)
    assert betas.shape == (num_diffusion_timesteps,)
    return betas


def get_named_beta_schedule(schedule_name, num_diffusion_timesteps):
    """
    Get a pre-defined beta schedule for the given name.
    The beta schedule library consists of beta schedules which remain similar
    in the limit of num_diffusion_timesteps.
    Beta schedules may be added, but should not be removed or changed once
    they are committed to maintain backwards compatibility.
    """
    if schedule_name == "linear":
        # Linear schedule from Ho et al, extended to work for any number of
        # diffusion steps.
        scale = 1000 / num_diffusion_timesteps
        return get_beta_schedule(
            "linear",
            beta_start=scale * 0.0001,
            beta_end=scale * 0.02,
            num_diffusion_timesteps=num_diffusion_timesteps,
        )
    elif schedule_name == "cosine":
        return betas_for_alpha_bar(
            num_diffusion_timesteps,
            lambda t: math.cos((t + 0.008) / 1.008 * math.pi / 2) ** 2,
        )
    else:
        raise NotImplementedError(f"unknown beta schedule: {schedule_name}")


def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    """
    Create a beta schedule that discretizes the given alpha_t_bar function,
    which defines the cumulative product of (1-beta) over time from t = [0,1].
    :param num_diffusion_timesteps: the number of betas to produce.
    :param alpha_bar: a lambda that takes an argument t from 0 to 1 and
                      produces the cumulative product of (1-beta) up to that
                      part of the diffusion process.
    :param max_beta: the maximum beta to use; use values lower than 1 to
                     prevent singularities.
    """
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


import math

get_named_beta_schedule("cosine", 1000)

cosine_betas = get_named_beta_schedule("cosine", 1000)
linear_betas = get_named_beta_schedule("linear", 1000)


def beta_to_alpha(beta):
    return (1 - beta).cumprod()


plt.plot(beta_to_alpha(cosine_betas), label="cosine alpha bar")
plt.plot(beta_to_alpha(linear_betas), label="linear alpha bar")
plt.legend()

In [ ]:
import torch

code = torch.tensor(
    [[4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4, 5, 1, 2, 1, 2, 1, 2, 1, 3, 4]]
)
mask = torch.ones_like(code, dtype=torch.bool)
z = torch.randn(1, code.shape[1], embedding_size) * 0.1  # torch.zeros(1, code.shape[1], embedding_size)

histogram = torch.tensor(
    [
        [
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
            [0.0, 4.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 3.0, 1.0, 1.0, 1.0],
            [0.0, 3.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 2.0, 1.0, 1.0, 1.0],
            [0.0, 2.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 1.0, 1.0, 1.0, 1.0],
            [0.0, 1.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 1.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 1.0, 1.0],
            [0.0, 0.0, 0.0, 0.0, 0.0, 1.0],
        ]
    ]
)

In [ ]:
from types import SimpleNamespace

import torch
import torch.optim as optim

from meds_torch.models.diffusion_utils.diffloss import DiffLoss


class HistogramNormalizer(torch.nn.Module):
    def __init__(self, h_token, o_token, num_bits=16):
        super().__init__()
        self.h_token = h_token
        self.o_token = o_token
        self.num_bits = num_bits
        # Create powers of 2 as a buffer to avoid recomputing
        self.register_buffer("powers", torch.pow(2, torch.arange(num_bits - 1, -1, -1).float()))

    def count_to_binary(self, count):
        """Convert number(s) to binary representation using PyTorch operations."""
        return ((count.unsqueeze(-1) // self.powers) % 2).to(torch.int)

    def binary_to_count(self, binary):
        return (binary * self.powers).sum(dim=-1, keepdim=True)

    def transform(self, x):
        # Zero out special tokens
        x[:, self.h_token] = 0
        x[:, self.o_token] = 0

        # Get counts and normalize histogram
        counts = x.sum(dim=-1, keepdim=True)
        normalized_hist = x / counts

        # Convert counts to binary representation
        binary_counts = self.count_to_binary(counts.squeeze(-1))

        # Concatenate normalized histogram with binary count
        data = torch.cat([normalized_hist, binary_counts], dim=-1)

        # Shift from [0,1] to [-1,1] range
        return data * 2 - 1

    def reverse_transform(self, x):
        # Shift from [-1,1] to [0,1]  range
        x = (x + 1) / 2

        # Clip to [0,1] range
        x = x.clip(min=0, max=1)
        # Split into histogram and binary count
        hist = x[:, : -(self.num_bits)]  # All but last num_bits dimensions
        binary = x[:, -(self.num_bits) :].round()  # Last num_bits dimensions

        x[:, self.h_token] = 0
        x[:, self.o_token] = 0

        # Convert binary back to count
        counts = self.binary_to_count(binary)

        # Scale histogram back up
        x = hist  # convert_to_counts_batched(hist / hist.sum(dim=-1, keepdim=True), counts)

        # Restore special tokens
        x[:, self.h_token] = 1
        x[:, self.o_token] = 1

        return x, counts


# Model and data setup
n_bits = 8
target_size = 6 + n_bits
embedding_size = 7
diffloss_w = 768
diffloss_d = 6
num_sampling_steps = "100"

diffusion = DiffLoss(
    target_channels=target_size,
    z_channels=embedding_size,
    width=diffloss_w,
    depth=diffloss_d,
    num_sampling_steps=num_sampling_steps,
    grad_checkpointing=False,
)

# Move model to GPU if available
histogram_normalizer = HistogramNormalizer(4, 5, n_bits)
histogram_normalizer

histogram = torch.tensor([[0.0, 4.0, 3.0, 1.0, 1.0, 1.0]])


histogram

In [ ]:
class SimpleDiffusion(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = torch.